In [19]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [20]:
import pandas as pd
import pprint

In [21]:
from eml_transformer.ingestion.sources.iem_afos import IEMAFOSSource

source = IEMAFOSSource(
    product_types=["AFD", ],
    wfos=["IND"],
    limit=3, 
)

raw = source.fetch_raw(
    from_date="2024-01-01",
    to_date="2025-01-05",
)

In [22]:
print("RAW RESPONSES:", len(raw))
print("RAW KEYS:", raw[0].keys())

RAW RESPONSES: 1
RAW KEYS: dict_keys(['pil', 'response'])


In [23]:
records = source.parse_records(raw)

In [24]:
print("PARSED RECORDS:", len(records))

for i, record in enumerate(records):
    print("\n" + "=" * 80)
    print(f"RECORD {i + 1}")
    print("=" * 80)

    raw_text = record["raw_text"]
    header = source._parse_header(raw_text)
    sections = source._parse_sections(raw_text)
    issued = source._extract_issued_text(raw_text)

    print("PIL:", record["pil"])
    print("HEADER:", header)
    print("ISSUED:", issued)
    print("SECTIONS:", list(sections.keys()))
    print("RAW LEN:", len(raw_text))

    print("\nFIRST 10 LINES:")
    print("\n".join(raw_text.splitlines()[:10]))

    print("\nKEY MESSAGES:")
    print(sections.get("KEY MESSAGES", "MISSING")[:1000])

PARSED RECORDS: 3

RECORD 1
PIL: AFDIND
HEADER: {'raw_id': '089', 'wmo': 'FXUS63', 'wmo_header': 'FXUS63 KIND 042325', 'office': 'KIND', 'issued_code': '042325', 'pil': 'AFDIND'}
ISSUED: Issued at 322 PM EST Sat Jan 4 2025
SECTIONS: ['KEY MESSAGES', 'SHORT TERM', 'LONG TERM', 'AVIATION', 'IND WATCHES/WARNINGS/ADVISORIES']
RAW LEN: 8494

FIRST 10 LINES:
089 
FXUS63 KIND 042325
AFDIND

Area Forecast Discussion
National Weather Service Indianapolis IN
625 PM EST Sat Jan 4 2025

.KEY MESSAGES...


KEY MESSAGES:
- Below normal temperatures expected for the foreseeable future. 
Bitterly cold conditions likely by mid week next week.

- WINTER STORM Warning in effect across much of central IN for a 
significant winter storm expected to impact the region Sunday - 
Monday.

- Winter Weather Advisory in effect for northern portions of central 
Indiana where lower snowfall totals and overall impacts are expected.

RECORD 2
PIL: AFDIND
HEADER: {'raw_id': '735', 'wmo': 'FXUS63', 'wmo_header': 'FXUS6

In [25]:
standardized = [
    source.standardize_record(record)
    for record in records
]

for i, record in enumerate(standardized):
    print("\n" + "=" * 80)
    print(f"STANDARDIZED {i + 1}")
    print("=" * 80)

    print("record_id:", record.record_id)
    print("title:", record.title)
    print("published_at:", record.published_at)
    print("region:", record.region)
    print("categories:", record.categories)
    print("text len:", len(record.text or ""))
    print("raw sections:", list(record.metadata.get("sections", {}).keys()))

    print("\nTEXT PREVIEW:")
    print((record.text or "")[:1000])


STANDARDIZED 1
record_id: 335f0848a9c6fa8b47488bd5b65c1d2ee149b2290a5166fe4c5676b10c506fca
title: AFD | KIND | Issued at 322 PM EST Sat Jan 4 2025
published_at: 2025-01-04T15:22:00-05:00
region: IND
categories: ['weather', 'nws', 'iem', 'afos', 'afd']
text len: 5531
raw sections: ['KEY MESSAGES', 'SHORT TERM', 'LONG TERM', 'AVIATION', 'IND WATCHES/WARNINGS/ADVISORIES']

TEXT PREVIEW:
- Below normal temperatures expected for the foreseeable future. 
Bitterly cold conditions likely by mid week next week.

- WINTER STORM Warning in effect across much of central IN for a 
significant winter storm expected to impact the region Sunday - 
Monday.

- Winter Weather Advisory in effect for northern portions of central 
Indiana where lower snowfall totals and overall impacts are expected.

Issued at 322 PM EST Sat Jan 4 2025

OVERVIEW -

1) WINTER STORM WARNING across much of central Indiana for a winter 
storm expected Sunday into Monday, with potential for significant to 
major impacts, in the

In [38]:
df = pd.read_parquet('../data/silver/source=iem_afos/records.parquet')

In [42]:
failed = df[
    ~df["published_at"].astype(str).str.match(
        r"^\d{4}-\d{2}-\d{2}T"
    )
]

row = failed[
    [
        "published_at",
        "title",
        "region",
        "raw",
    ]
].iloc[0]

pp = pprint.PrettyPrinter(
    indent=2,
    width=120,
    sort_dicts=False,
)

pp.pprint(row.to_dict())

{ 'published_at': '250 pM CDT Mon Jun 30 2025',
  'title': 'HWO | KILX | 250 pM CDT Mon Jun 30 2025',
  'region': 'ILX',
  'raw': '890 \n'
         'FLUS43 KILX 301950\n'
         'HWOILX\n'
         '\n'
         'Hazardous Weather Outlook\n'
         'National Weather Service Lincoln IL\n'
         '250 pM CDT Mon Jun 30 2025\n'
         '\n'
         'ILZ027>031-036>038-040>057-061>063-066>068-071>073-011015-\n'
         'Knox-Stark-Peoria-Marshall-Woodford-Fulton-Tazewell-McLean-Schuyler-\n'
         'Mason-Logan-De Witt-Piatt-Champaign-Vermilion-Cass-Menard-Scott-\n'
         'Morgan-Sangamon-Christian-Macon-Moultrie-Douglas-Coles-Edgar-Shelby-\n'
         'Cumberland-Clark-Effingham-Jasper-Crawford-Clay-Richland-Lawrence-\n'
         '250 pM CDT Mon Jun 30 2025\n'
         '\n'
         'This Hazardous Weather Outlook is for central Illinois, east \n'
         'central Illinois, southeast Illinois and west central Illinois.\n'
         '\n'
         '.DAY ONE...This afternoon and

In [37]:
print(len(failed))

111


In [44]:
failed['title']

31555              HWO | KILX | 250 pM CDT Mon Jun 30 2025
31556    AFD | KIND | Issued at 9265 AM EDT Sat Apr 19 ...
31557    HWO | KLSX | Issued by National Weather Servic...
31558    HWO | KLSX | Issued by National Weather Servic...
31559    HWO | KLSX | Issued by National Weather Servic...
                               ...                        
31661    HWO | KIWX | Issued by National Weather Servic...
31662    HWO | KLOT | Issued by National Weather Servic...
31663    WSW | KIWX | Issued by National Weather Servic...
31664    HWO | KIND | Issued by National Weather Servic...
31665    HWO | KILX | Issued by National Weather Servic...
Name: title, Length: 111, dtype: object

In [40]:
df.columns

Index(['record_id', 'source', 'source_type', 'title', 'text', 'published_at',
       'retrieved_at', 'url', 'region', 'categories', 'metadata', 'raw'],
      dtype='object')

In [45]:
for _, row in failed.head(5).iterrows():
    print("\n" + "=" * 100)

    print("TITLE:")
    print(row["title"])

    print("\nPUBLISHED_AT:")
    print(row["published_at"])

    print("\nREGION:")
    print(row["region"])

    print("\nRAW:")
    pprint.pp(row["raw"])

    print("=" * 100)


TITLE:
HWO | KILX | 250 pM CDT Mon Jun 30 2025

PUBLISHED_AT:
250 pM CDT Mon Jun 30 2025

REGION:
ILX

RAW:
('890 \n'
 'FLUS43 KILX 301950\n'
 'HWOILX\n'
 '\n'
 'Hazardous Weather Outlook\n'
 'National Weather Service Lincoln IL\n'
 '250 pM CDT Mon Jun 30 2025\n'
 '\n'
 'ILZ027>031-036>038-040>057-061>063-066>068-071>073-011015-\n'
 'Knox-Stark-Peoria-Marshall-Woodford-Fulton-Tazewell-McLean-Schuyler-\n'
 'Mason-Logan-De Witt-Piatt-Champaign-Vermilion-Cass-Menard-Scott-\n'
 'Morgan-Sangamon-Christian-Macon-Moultrie-Douglas-Coles-Edgar-Shelby-\n'
 'Cumberland-Clark-Effingham-Jasper-Crawford-Clay-Richland-Lawrence-\n'
 '250 pM CDT Mon Jun 30 2025\n'
 '\n'
 'This Hazardous Weather Outlook is for central Illinois, east \n'
 'central Illinois, southeast Illinois and west central Illinois.\n'
 '\n'
 '.DAY ONE...This afternoon and tonight.\n'
 '\n'
 'Heat indices will peak in the middle 90s this afternoon.\n'
 '\n'
 'Scattered thunderstorms are expected late this afternoon and \n'
 'evening,

In [47]:
text = '435 AM CST Wed Feb 12 2025'

In [ ]:
FORMATS = [
    "%I%M %p %Z %a %b %d %Y",
    "%H%M %Z %a %b %d %Y",
    "%I%M %p %Z %b %d %Y",
]